In [ ]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import cv2
import mediapipe as mp

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

GESTURES = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'L']
SEQUENCE_LENGTH = 30
NUM_SEQUENCES = 60
DATA_PATH = 'asl_data'
MODEL_PATH = 'asl_lstm_model.keras'
CAMERA_INDEX = 0

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_draw_styles = mp.solutions.drawing_styles

print(f"TensorFlow: {tf.__version__}")
print(f"MediaPipe: {mp.__version__}")
print(f"OpenCV: {cv2.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU: {gpus if gpus else 'None - using CPU'}") 

In [ ]:
def mediapipe_detection(image, model):
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results


def draw_landmarks(image, results):
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                image,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_draw_styles.get_default_hand_landmarks_style(),
                mp_draw_styles.get_default_hand_connections_style()
            )
    return image


def extract_keypoints(results):
    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]
        keypoints = np.array([[lm.x, lm.y, lm.z] for lm in hand.landmark]).flatten()
    else:
        keypoints = np.zeros(21 * 3)
    return keypoints


def normalize_keypoints(keypoints):
    if np.all(keypoints == 0):
        return keypoints
    kp = keypoints.reshape(21, 3)
    wrist = kp[0].copy()
    kp = kp - wrist
    max_val = np.max(np.abs(kp))
    if max_val > 0:
        kp = kp / max_val
    return kp.flatten()

In [ ]:
for gesture in GESTURES:
    for seq_idx in range(NUM_SEQUENCES):
        folder = os.path.join(DATA_PATH, gesture, str(seq_idx))
        os.makedirs(folder, exist_ok=True)

print("Folders created")

In [ ]:
def put_text(frame, text, pos=(10, 40), scale=1.2, color=(0, 255, 0), thickness=2):
    cv2.putText(frame, text, pos, cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)


def collect_data():
    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    if not cap.isOpened():
        print("Cannot open webcam")
        return

    with mp_hands.Hands(
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.7,
        min_tracking_confidence=0.5
    ) as hands:

        for gesture in GESTURES:
            for seq_idx in range(NUM_SEQUENCES):

                for countdown in range(2, 0, -1):
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame = cv2.flip(frame, 1)
                    put_text(frame, f"Next: '{gesture}' | Seq {seq_idx+1}/{NUM_SEQUENCES}",
                             pos=(10, 40), color=(0, 200, 255))
                    put_text(frame, f"Starting in {countdown}...",
                             pos=(10, 85), scale=0.9, color=(255, 255, 0))
                    cv2.imshow('ASL Data Collection', frame)
                    cv2.waitKey(1000)

                for frame_idx in range(SEQUENCE_LENGTH):
                    ret, frame = cap.read()
                    if not ret:
                        break

                    frame = cv2.flip(frame, 1)
                    frame, results = mediapipe_detection(frame, hands)
                    frame = draw_landmarks(frame, results)

                    keypoints = extract_keypoints(results)
                    keypoints = normalize_keypoints(keypoints)

                    npy_path = os.path.join(DATA_PATH, gesture, str(seq_idx), str(frame_idx))
                    np.save(npy_path, keypoints)

                    progress = int((frame_idx / SEQUENCE_LENGTH) * 200)
                    cv2.rectangle(frame, (10, 450), (10 + progress, 465), (0, 255, 0), -1)
                    cv2.rectangle(frame, (10, 450), (210, 465), (200, 200, 200), 1)
                    put_text(frame, f"RECORDING '{gesture}' [{frame_idx+1}/{SEQUENCE_LENGTH}]",
                             pos=(10, 40), color=(0, 255, 0))
                    put_text(frame, f"Sequence {seq_idx+1} of {NUM_SEQUENCES} | Press Q to quit",
                             pos=(10, 85), scale=0.7, color=(200, 200, 200))

                    cv2.imshow('ASL Data Collection', frame)

                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        cap.release()
                        cv2.destroyAllWindows()
                        print("Collection stopped")
                        return

                print(f"Gesture '{gesture}' | Sequence {seq_idx+1}/{NUM_SEQUENCES} saved")

    cap.release()
    cv2.destroyAllWindows()
    print("Data collection complete")


collect_data()

In [ ]:
sequences = []
labels = []

for gesture in GESTURES:
    gesture_count = 0
    for seq_idx in range(NUM_SEQUENCES):
        window = []
        missing = False

        for frame_idx in range(SEQUENCE_LENGTH):
            npy_path = os.path.join(DATA_PATH, gesture, str(seq_idx), f"{frame_idx}.npy")
            if not os.path.exists(npy_path):
                missing = True
                break
            window.append(np.load(npy_path))

        if not missing and len(window) == SEQUENCE_LENGTH:
            sequences.append(window)
            labels.append(gesture)
            gesture_count += 1

    print(f"'{gesture}': {gesture_count} sequences loaded")

X = np.array(sequences)
print(f"X shape: {X.shape}")

In [ ]:
le = LabelEncoder()
y_int = le.fit_transform(labels)
y = to_categorical(y_int, num_classes=len(GESTURES))

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y_int
)

print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}")

In [ ]:
sample_idx = 0
sample = X_train[sample_idx]
label_name = le.inverse_transform([np.argmax(y_train[sample_idx])])[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for lm_idx in range(21):
    axes[0].plot(sample[:, lm_idx * 3], alpha=0.6, linewidth=1)
axes[0].set_title(f"X-coordinates | Gesture: '{label_name}'")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Normalised x")
axes[0].grid(alpha=0.3)

for lm_idx in range(21):
    axes[1].plot(sample[:, lm_idx * 3 + 1], alpha=0.6, linewidth=1)
axes[1].set_title(f"Y-coordinates | Gesture: '{label_name}'")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Normalised y")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def build_model(sequence_length, num_features, num_classes):
    model = Sequential(name="ASL_LSTM", layers=[
        LSTM(128, return_sequences=True,
             input_shape=(sequence_length, num_features),
             kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(64, return_sequences=True,
             kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        Dropout(0.3),

        Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_model(
    sequence_length=SEQUENCE_LENGTH,
    num_features=X_train.shape[2],
    num_classes=len(GESTURES)
)

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print(f"Model saved to '{MODEL_PATH}'")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2, linestyle='--')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2, color='orange')
axes[1].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--', color='red')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
best_model = load_model(MODEL_PATH)

val_loss, val_acc = best_model.evaluate(X_val, y_val, verbose=0)
print(f"Val Loss: {val_loss:.4f}  |  Val Accuracy: {val_acc*100:.1f}%")

y_pred_prob = best_model.predict(X_val, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_val, axis=1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

tick_marks = np.arange(len(GESTURES))
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(le.classes_, fontsize=11)
ax.set_yticklabels(le.classes_, fontsize=11)

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=12)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=le.classes_))

In [ ]:
from collections import deque

inference_model = load_model(MODEL_PATH)

CONFIDENCE_THRESHOLD = 0.75
SMOOTH_WINDOW = 5


def draw_confidence_bar(frame, confidence, x=10, y=120, width=200, height=18):
    cv2.rectangle(frame, (x, y), (x + width, y + height), (50, 50, 50), -1)
    fill = int(width * confidence)
    color = (0, int(255 * confidence), int(255 * (1 - confidence)))
    cv2.rectangle(frame, (x, y), (x + fill, y + height), color, -1)
    cv2.rectangle(frame, (x, y), (x + width, y + height), (200, 200, 200), 1)
    cv2.putText(frame, f"{confidence*100:.0f}%",
                (x + width + 8, y + 13),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)


def run_inference():
    sequence = deque(maxlen=SEQUENCE_LENGTH)
    predictions = deque(maxlen=SMOOTH_WINDOW)

    cap = cv2.VideoCapture(CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)

    if not cap.isOpened():
        print("Cannot open webcam")
        return

    prev_time = time.time()
    current_pred = ""
    current_conf = 0.0

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.7,
        min_tracking_confidence=0.5
    ) as hands:

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            frame, results = mediapipe_detection(frame, hands)
            frame = draw_landmarks(frame, results)

            keypoints = extract_keypoints(results)
            keypoints = normalize_keypoints(keypoints)
            sequence.append(keypoints)

            if len(sequence) == SEQUENCE_LENGTH:
                input_seq = np.expand_dims(np.array(sequence), axis=0)
                probs = inference_model.predict(input_seq, verbose=0)[0]
                pred_idx = np.argmax(probs)
                confidence = float(probs[pred_idx])

                if confidence >= CONFIDENCE_THRESHOLD:
                    predictions.append(pred_idx)

                if predictions:
                    smoothed_idx = max(set(predictions), key=list(predictions).count)
                    current_pred = le.inverse_transform([smoothed_idx])[0]
                    current_conf = confidence

            if current_pred:
                cv2.putText(frame, f"ASL: {current_pred}",
                            (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 0), 3)
                draw_confidence_bar(frame, current_conf, x=10, y=80)
            else:
                cv2.putText(frame, "Show a hand...",
                            (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (100, 100, 100), 2)

            now = time.time()
            fps = 1.0 / (now - prev_time + 1e-6)
            prev_time = now
            cv2.putText(frame, f"FPS: {fps:.1f}",
                        (10, frame.shape[0] - 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

            buf_pct = int((len(sequence) / SEQUENCE_LENGTH) * 200)
            cv2.rectangle(frame, (10, frame.shape[0] - 20), (10 + buf_pct, frame.shape[0] - 8),
                          (0, 200, 255), -1)
            cv2.rectangle(frame, (10, frame.shape[0] - 20), (210, frame.shape[0] - 8),
                          (150, 150, 150), 1)
            cv2.putText(frame, "Buffer",
                        (215, frame.shape[0] - 9),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1)
            cv2.putText(frame, "Press Q to quit",
                        (frame.shape[1] - 180, frame.shape[0] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

            cv2.imshow('ASL Real-Time Recognition', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print("Inference stopped")


run_inference()